# 06B - Decision Tree Classifier

Enterprise decision tree model with evaluation and interpretation.

In [1]:

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report,confusion_matrix

df=pd.read_csv(r"/mnt/data/american_bankruptcy.csv")
df['target']=df['status_label'].map({'alive':0,'failed':1})
drop=['status_label','target']
if 'company_name' in df.columns:
    drop.append('company_name')
X=df.drop(columns=drop)
y=df['target']

num=X.select_dtypes(include='number').columns
cat=X.select_dtypes(exclude='number').columns

pre=ColumnTransformer([
('num',SimpleImputer(strategy='median'),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                 ('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)
])

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


## Train Decision Tree

In [2]:

model=Pipeline([
('preprocessor',pre),
('classifier',DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42))
])
model.fit(X_train,y_train)
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]


## Cross Validation

In [3]:

scores=cross_val_score(model,X_train,y_train,cv=5,scoring='f1')
print(scores)
print("Mean F1:",scores.mean().round(4))


[0.19704275 0.22283248 0.20716321 0.20048561 0.19989801]
Mean F1: 0.2055


## Model Evaluation

In [4]:

results=pd.DataFrame({
'Metric':['Accuracy','Precision','Recall','F1','ROC-AUC'],
'Value':[
accuracy_score(y_test,pred),
precision_score(y_test,pred),
recall_score(y_test,pred),
f1_score(y_test,pred),
roc_auc_score(y_test,proba)
]})
display(results)
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))


,Metric,Value
0,Accuracy,0.604499
1,Precision,0.113086
2,Recall,0.725096
3,F1,0.195658
4,ROC-AUC,0.724936


              precision    recall  f1-score   support

           0       0.97      0.60      0.74     14693
           1       0.11      0.73      0.20      1044

    accuracy                           0.60     15737
   macro avg       0.54      0.66      0.47     15737
weighted avg       0.91      0.60      0.70     15737

[[8756 5937]
 [ 287  757]]


## Feature Importance

In [5]:

clf=model.named_steps['classifier']
feat=list(num)
if len(cat)>0:
    feat.extend(model.named_steps['preprocessor'].named_transformers_['cat']
                .named_steps['oh'].get_feature_names_out(cat))
imp=pd.DataFrame({'Feature':feat,'Importance':clf.feature_importances_})
imp=imp.sort_values('Importance',ascending=False)
display(imp.head(20))


,Feature,Importance
0,year,0.207059
6,X6,0.175473
8,X8,0.142181
3,X3,0.075677
11,X11,0.072502
7,X7,0.057012
17,X17,0.033958
13,X13,0.031967
5,X5,0.030608
15,X15,0.025710


## Save Model

In [6]:

joblib.dump(model,"decision_tree_model.joblib")
print("Saved decision_tree_model.joblib")


Saved decision_tree_model.joblib


## Executive Summary

In [7]:

print("• Decision Trees capture nonlinear relationships.")
print("• Compare these metrics against the Logistic Regression baseline.")
print("• Inspect feature importance before progressing to ensemble models.")


• Decision Trees capture nonlinear relationships.
• Compare these metrics against the Logistic Regression baseline.
• Inspect feature importance before progressing to ensemble models.
